In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/uzairsidd/cyber-sentry-dataset/UNSW_NB15_testing-set.csv
/kaggle/input/datasets/uzairsidd/cyber-sentry-dataset/UNSW_NB15_training-set.csv


# CyberSentryFusion

## Data Preprocessing and Model Training

In this notebook, we'll prepare the dataset, process numerical and categorical features, train a machine learning model, and evaluate its performance in detecting network attacks.


In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_auc_score
)

import joblib

In [7]:
train_path = "/kaggle/input/datasets/uzairsidd/cyber-sentry-dataset/UNSW_NB15_training-set.csv"
test_path = "/kaggle/input/datasets/uzairsidd/cyber-sentry-dataset/UNSW_NB15_testing-set.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

Training shape: (175341, 45)
Testing shape: (82332, 45)


In [8]:
X_train = train_df.drop(columns=["label", "attack_cat", "id"])
y_train = train_df["label"]

X_test = test_df.drop(columns=["label", "attack_cat", "id"])
y_test = test_df["label"]

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Target distribution:\n", y_train.value_counts())

Training features: (175341, 42)
Testing features: (82332, 42)
Target distribution:
 label
1    119341
0     56000
Name: count, dtype: int64


In [9]:
# Identify categorical and numerical features
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical features:", categorical_features)
print("Number of categorical features:", len(categorical_features))

print("\nNumber of numerical features:", len(numerical_features))
print("Numerical features:", numerical_features)

Categorical features: ['proto', 'service', 'state']
Number of categorical features: 3

Number of numerical features: 39
Numerical features: ['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports']


In [10]:
# Preprocess numerical and categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [11]:
# Create a pipeline with preprocessing and Logistic Regression
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

print("Model pipeline created successfully.")

Model pipeline created successfully.


## Model Comparison

We'll compare Logistic Regression, Decision Tree, Random Forest, and XGBoost on the same training and testing datasets. We'll evaluate their performance using accuracy, precision, recall, F1-score, and ROC-AUC to select a suitable model for network attack detection.


In [12]:
try:
    import xgboost as xgb
    print("XGBoost version:", xgb.__version__)
except ImportError:
    print("XGBoost is not installed.")

XGBoost version: 3.4.1


In [13]:
# Imports
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Identify feature types
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    exclude=["object"]
).columns.tolist()

# Preprocessing
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

# Define models
models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "Decision Tree": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "XGBoost": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        ))
    ])
}

# Train models
for name, model_pipeline in models.items():
    print(f"Training {name}...")
    model_pipeline.fit(X_train, y_train)

print("\nAll models trained successfully!")

Training Logistic Regression...
Training Decision Tree...
Training Random Forest...
Training XGBoost...

All models trained successfully!


In [14]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

results = []

for name, model_pipeline in models.items():
    # Predict classes and attack probabilities
    y_pred = model_pipeline.predict(X_test)
    y_prob = model_pipeline.predict_proba(X_test)[:, 1]

    # Calculate evaluation metrics
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

    print(f"\n{name}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

# Display comparison table
results_df = pd.DataFrame(results).set_index("Model")
results_df = results_df.sort_values(by="F1-Score", ascending=False)

print("\nModel Comparison:")
display(results_df.round(4))


Logistic Regression
Confusion Matrix:
[[26571 10429]
 [ 3115 42217]]

Decision Tree
Confusion Matrix:
[[27165  9835]
 [ 1644 43688]]

Random Forest
Confusion Matrix:
[[26960 10040]
 [  626 44706]]

XGBoost
Confusion Matrix:
[[27048  9952]
 [  517 44815]]

Model Comparison:


,Accuracy,Precision,Recall,F1-Score,ROC-AUC
Model,,,,,
XGBoost,0.8728,0.8183,0.9886,0.8954,0.9844
Random Forest,0.8705,0.8166,0.9862,0.8934,0.9789
Decision Tree,0.8606,0.8162,0.9637,0.8839,0.8507
Logistic Regression,0.8355,0.8019,0.9313,0.8618,0.9561


In [15]:
from sklearn.metrics import confusion_matrix

# Get attack probabilities from the trained XGBoost model
xgb_model = models["XGBoost"]
y_prob = xgb_model.predict_proba(X_test)[:, 1]

# Compare different classification thresholds
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_threshold = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test, y_pred_threshold
    ).ravel()

    recall = tp / (tp + fn)
    false_positive_rate = fp / (fp + tn)
    precision = tp / (tp + fp)

    print(f"\nThreshold: {threshold}")
    print(f"Recall: {recall:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"False Positive Rate: {false_positive_rate:.4f}")
    print(f"False Negatives: {fn}")
    print(f"False Positives: {fp}")


Threshold: 0.3
Recall: 0.9991
Precision: 0.7490
False Positive Rate: 0.4102
False Negatives: 40
False Positives: 15179

Threshold: 0.4
Recall: 0.9965
Precision: 0.7722
False Positive Rate: 0.3602
False Negatives: 158
False Positives: 13328

Threshold: 0.5
Recall: 0.9886
Precision: 0.8183
False Positive Rate: 0.2690
False Negatives: 517
False Positives: 9952

Threshold: 0.6
Recall: 0.9723
Precision: 0.8646
False Positive Rate: 0.1865
False Negatives: 1257
False Positives: 6900

Threshold: 0.7
Recall: 0.9541
Precision: 0.8997
False Positive Rate: 0.1302
False Negatives: 2082
False Positives: 4819


In [16]:
from sklearn.model_selection import train_test_split

# Split the original training data: 80% training, 20% validation
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print("Training samples:", X_train_split.shape[0])
print("Validation samples:", X_val.shape[0])

print("\nTraining class distribution:")
print(y_train_split.value_counts())

print("\nValidation class distribution:")
print(y_val.value_counts())

Training samples: 140272
Validation samples: 35069

Training class distribution:
label
1    95472
0    44800
Name: count, dtype: int64

Validation class distribution:
label
1    23869
0    11200
Name: count, dtype: int64


In [17]:
from sklearn.base import clone

# Create a fresh copy of the XGBoost pipeline
xgb_tuning = clone(models["XGBoost"])

# Train only on the training split
xgb_tuning.fit(X_train_split, y_train_split)

print("XGBoost trained successfully on the training split!")

XGBoost trained successfully on the training split!


In [18]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Predict attack probabilities on validation data
y_val_prob = xgb_tuning.predict_proba(X_val)[:, 1]

# Use the default threshold of 0.5
y_val_pred = (y_val_prob >= 0.5).astype(int)

print("Classification Report:")
print(classification_report(y_val, y_val_pred, target_names=["Normal", "Attack"]))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

print("Validation ROC-AUC:", round(roc_auc_score(y_val, y_val_prob), 4))

Classification Report:
              precision    recall  f1-score   support

      Normal       0.95      0.91      0.93     11200
      Attack       0.96      0.98      0.97     23869

    accuracy                           0.95     35069
   macro avg       0.95      0.94      0.95     35069
weighted avg       0.95      0.95      0.95     35069

Confusion Matrix:
[[10167  1033]
 [  560 23309]]
Validation ROC-AUC: 0.9923


In [19]:
from sklearn.metrics import confusion_matrix

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

for threshold in thresholds:
    y_pred = (y_val_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) else 0
    )
    fpr = fp / (fp + tn)

    print(f"\nThreshold: {threshold}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")
    print(f"False Positive Rate: {fpr:.4f}")
    print(f"False Positives: {fp}")
    print(f"False Negatives: {fn}")


Threshold: 0.3
Precision: 0.9212
Recall: 0.9981
F1-score: 0.9581
False Positive Rate: 0.1820
False Positives: 2038
False Negatives: 45

Threshold: 0.4
Precision: 0.9383
Recall: 0.9909
F1-score: 0.9639
False Positive Rate: 0.1388
False Positives: 1554
False Negatives: 217

Threshold: 0.5
Precision: 0.9576
Recall: 0.9765
F1-score: 0.9670
False Positive Rate: 0.0922
False Positives: 1033
False Negatives: 560

Threshold: 0.6
Precision: 0.9720
Recall: 0.9554
F1-score: 0.9636
False Positive Rate: 0.0587
False Positives: 658
False Negatives: 1065

Threshold: 0.7
Precision: 0.9806
Recall: 0.9330
F1-score: 0.9562
False Positive Rate: 0.0394
False Positives: 441
False Negatives: 1600


In [20]:
# Selected using validation results
best_threshold = 0.5

print("Selected threshold:", best_threshold)

Selected threshold: 0.5


In [21]:
# Create a fresh XGBoost pipeline
final_model = clone(models["XGBoost"])

# Train on the complete original training dataset
final_model.fit(X_train, y_train)

print("Final XGBoost model trained successfully!")
print("Selected threshold:", best_threshold)

Final XGBoost model trained successfully!
Selected threshold: 0.5


In [22]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_auc_score
)

# Predict attack probabilities
y_test_prob = final_model.predict_proba(X_test)[:, 1]

# Apply the selected threshold
y_test_pred = (y_test_prob >= best_threshold).astype(int)

# Evaluate the model
print("Classification Report:")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=["Normal", "Attack"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("Accuracy:", round(accuracy_score(y_test, y_test_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_test_prob), 4))

Classification Report:
              precision    recall  f1-score   support

      Normal       0.98      0.73      0.84     37000
      Attack       0.82      0.99      0.90     45332

    accuracy                           0.87     82332
   macro avg       0.90      0.86      0.87     82332
weighted avg       0.89      0.87      0.87     82332

Confusion Matrix:
[[27048  9952]
 [  517 44815]]
Accuracy: 0.8728
ROC-AUC: 0.9844


In [23]:
import joblib
import os

# Create a directory for saved models
os.makedirs("/kaggle/working/models", exist_ok=True)

# Save the trained pipeline
joblib.dump(
    final_model,
    "/kaggle/working/models/xgboost_model.joblib"
)

# Save the selected classification threshold
joblib.dump(
    best_threshold,
    "/kaggle/working/models/threshold.joblib"
)

print("Model saved successfully!")
print("Model path: /kaggle/working/models/xgboost_model.joblib")
print("Threshold:", best_threshold)

Model saved successfully!
Model path: /kaggle/working/models/xgboost_model.joblib
Threshold: 0.5


In [24]:
import os
import joblib

model_path = "/kaggle/working/models/xgboost_model.joblib"
threshold_path = "/kaggle/working/models/threshold.joblib"

# Verify files exist
print("Model exists:", os.path.exists(model_path))
print("Threshold exists:", os.path.exists(threshold_path))

# Load the saved files
loaded_model = joblib.load(model_path)
loaded_threshold = joblib.load(threshold_path)

print("Model loaded successfully!")
print("Loaded threshold:", loaded_threshold)


Model exists: True
Threshold exists: True
Model loaded successfully!
Loaded threshold: 0.5


In [25]:
import shutil

shutil.make_archive(
    "/kaggle/working/cybersentry_model",
    "zip",
    "/kaggle/working/models"
)

print("ZIP created: /kaggle/working/cybersentry_model.zip")

ZIP created: /kaggle/working/cybersentry_model.zip
